# Generating Dataset for FB_INF project

The current notebook is a simple way of randomly selecting images from the ImageNet dataset for the desired classes in the FB_INF experiment for Paulo's first research project.

The code presented here is super simple. I have it on a separate notebook for ease of use and organization

In [10]:
#Loading the directories needed

%load_ext autoreload
%autoreload 2

import os
import shutil
import numpy as np

from wormholes import *
from wormholes.perturb import *
from wormholes.perturb.gen_v3 import GenV3

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
#When working with the jupyter network and the cluster, it sometimes fails to find the right directories. 
# I thus ensure that they are properly set

# Set the HOME environment variable to your current working directory
os.environ["HOME"] = "/project/3018078.01/Gaziv/Wormholes_FB"

# Now the `os.path.expanduser("~")` will resolve to this directory
print(os.path.expanduser("~")) 

/project/3018078.01/Gaziv/Wormholes_FB


In [12]:

#Selecting the dataset I want to use for trasnferring the images from ilrsvc to my own folders
class_dict = CLASS_DICT['RestrictedImageNet'] #maps class labels to human-readble names for Restricted ImageNet

#because the dictionary containing the info to pass the RIN and imagenet images to my folders is a function of the class Genv3
#I am creating an instance of the class to get to use the features
instance = GenV3()

# Access the data dictionary
data_dict = instance.data_dict

# Print or inspect data_dict (not needed, but useful)
print(data_dict.keys())  # To see available classes
print(len(data_dict))  # To see how many classes there are

#Inspect where the images come from from all the imagenet classes
# print(data_dict['frog'])

dict_keys(['frog', 'turtle', 'lizard', 'bird', 'crab', 'dog', 'cat', 'bear', 'insect', 'rabbit', 'gazelle', 'primate'])
12


In [ ]:
#define the directory for the output (i.e., where the images will get saved)
output_folder_root = "/project/3018078.01/Gaziv/Wormholes_FB/data/OOD_Paulo"

#make a function to sample images. (not fully my function). 
#for the specific n_sample_class, you will always get the same images if ask for the same number. 
#I ensure that every (class, sample) pair gets a unique seed.
# i (class index), n_sample_class (the number of images to select per class), j (the image index within the class (ranges from 0 to n_sample_class - 1)).

def sample_images(output_dir=output_folder_root, n_sample_class=1):
    """Randomly samples n_sample_class images per class and saves them to output_dir."""
    os.makedirs(output_dir, exist_ok=True)

    for i, (class_name, image_paths) in enumerate(data_dict.items()):
        class_output_dir = os.path.join(output_dir, class_name)
        os.makedirs(class_output_dir, exist_ok=True)

        rng = np.random.RandomState(i)

        # Sample without replacement to ensure diverse selection
        sampled_paths = rng.choice(image_paths, size=min(n_sample_class, len(image_paths)), replace=False)

        for img_path in sampled_paths:
            shutil.copy(img_path, os.path.join(class_output_dir, os.path.basename(img_path)))
            print(f"Copied {img_path} -> {class_output_dir}")

# Example Usage
# sample_images(n_sample_class=150)


Copied /project/3018078.01/Gaziv/Wormholes_FB/data/ilsvrc/val/n01644900/ILSVRC2012_val_00028673.JPEG -> /project/3018078.01/Gaziv/Wormholes_FB/data/OOD_Paulo/frog
Copied /project/3018078.01/Gaziv/Wormholes_FB/data/ilsvrc/val/n01644900/ILSVRC2012_val_00039478.JPEG -> /project/3018078.01/Gaziv/Wormholes_FB/data/OOD_Paulo/frog
Copied /project/3018078.01/Gaziv/Wormholes_FB/data/ilsvrc/val/n01644373/ILSVRC2012_val_00013201.JPEG -> /project/3018078.01/Gaziv/Wormholes_FB/data/OOD_Paulo/frog
Copied /project/3018078.01/Gaziv/Wormholes_FB/data/ilsvrc/val/n01644900/ILSVRC2012_val_00044993.JPEG -> /project/3018078.01/Gaziv/Wormholes_FB/data/OOD_Paulo/frog
Copied /project/3018078.01/Gaziv/Wormholes_FB/data/ilsvrc/val/n01644373/ILSVRC2012_val_00031598.JPEG -> /project/3018078.01/Gaziv/Wormholes_FB/data/OOD_Paulo/frog
Copied /project/3018078.01/Gaziv/Wormholes_FB/data/ilsvrc/val/n01644900/ILSVRC2012_val_00005768.JPEG -> /project/3018078.01/Gaziv/Wormholes_FB/data/OOD_Paulo/frog
Copied /project/301807

In [9]:

for i, (class_name, image_paths) in enumerate(data_dict.items()):
    print(len(image_paths))

data_root = instance.data_root
ds = RestrictedImageNet(f"{data_root}/ilsvrc")
folder_ds = ImageFolder(root=f"{ds.data_path}/val", label_mapping=ds.label_mapping)
# print(folder_ds.samples[:20])  # Check if all early samples belong to the same few classes
print({k: len(v) for k, v in data_dict.items()})  # Check how many images each superclass has



150
150
150
150
150
150
150
150
150
150
150
150
{'frog': 150, 'turtle': 150, 'lizard': 150, 'bird': 150, 'crab': 150, 'dog': 150, 'cat': 150, 'bear': 150, 'insect': 150, 'rabbit': 150, 'gazelle': 150, 'primate': 150}


After I have generated the images for all classes and then went through them manually in which I made sure no more than one animal was present, no text and the quality of the images was fine. 
The following line of code is to get rid of all the other images that were not selected from those folders.

In [16]:
#define the directory of the dataset (i.e., where the images are saved)
dataset_folder_root = "/project/3018078.01/Gaziv/Wormholes_FB/data/OOD_Paulo"

def clean_folders(data_dir=dataset_folder_root, prefix="ILSVRC", dry_run=True):
    """
    Removes files in the dataset folder that start with the given prefix.

    Parameters:
        data_dir (str or Path): Root directory to clean.
        prefix (str): Filename prefix to match for deletion.
        dry_run (bool): If True, prints files to be deleted instead of deleting them.
    """

    data_dir = Path(data_dir)  # Ensure it's a Path object
    if not data_dir.exists():
        print(f"Error: Directory '{data_dir}' does not exist.")
        return

    # Traverse directory tree
    for file_path in data_dir.rglob(f"{prefix}*"):  # More efficient than os.walk() according to new python websites
        if file_path.is_file():
            if dry_run:
                print(f"Would delete: {file_path}")  # Safety check
            else:
                file_path.unlink()  # Delete file
                print(f"Deleted: {file_path}")

# Run the function in dry-run mode first
clean_folders(dry_run=False)



Deleted: /project/3018078.01/Gaziv/Wormholes_FB/data/OOD_Paulo/frog/ILSVRC2012_val_00039478.JPEG
Deleted: /project/3018078.01/Gaziv/Wormholes_FB/data/OOD_Paulo/frog/ILSVRC2012_val_00044993.JPEG
Deleted: /project/3018078.01/Gaziv/Wormholes_FB/data/OOD_Paulo/frog/ILSVRC2012_val_00031598.JPEG
Deleted: /project/3018078.01/Gaziv/Wormholes_FB/data/OOD_Paulo/frog/ILSVRC2012_val_00018879.JPEG
Deleted: /project/3018078.01/Gaziv/Wormholes_FB/data/OOD_Paulo/frog/ILSVRC2012_val_00030352.JPEG
Deleted: /project/3018078.01/Gaziv/Wormholes_FB/data/OOD_Paulo/frog/ILSVRC2012_val_00044855.JPEG
Deleted: /project/3018078.01/Gaziv/Wormholes_FB/data/OOD_Paulo/frog/ILSVRC2012_val_00033863.JPEG
Deleted: /project/3018078.01/Gaziv/Wormholes_FB/data/OOD_Paulo/frog/ILSVRC2012_val_00040420.JPEG
Deleted: /project/3018078.01/Gaziv/Wormholes_FB/data/OOD_Paulo/frog/ILSVRC2012_val_00032966.JPEG
Deleted: /project/3018078.01/Gaziv/Wormholes_FB/data/OOD_Paulo/frog/ILSVRC2012_val_00043191.JPEG
Deleted: /project/3018078.01/G